## export 3D UMAP coordinates 

- last updated: 9/30/2025

- export the 3D UMAP coordinates and metadata for 3D UMAP visualizer by Kyle

- the metadata columns are the following:
    - chromosome
    - peak_type (Argelaguet)
    - (the most accessible) celltype
    - (the most accessible) timepoint
    - motif families?

In [15]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import sys
import os

# rapids-singlecell
import cupy as cp
import rapids_singlecell as rsc

In [16]:
# figure parameter setting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
mpl.rcParams.update(mpl.rcParamsDefault) #Reset rcParams to default

# Editable text and proper LaTeX fonts in illustrator
# matplotlib.rcParams['ps.useafm'] = True
# Editable fonts. 42 is the magic number
mpl.rcParams['pdf.fonttype'] = 42
sns.set(style='whitegrid', context='paper')
# Set default DPI for saved figures
mpl.rcParams['savefig.dpi'] = 600

In [17]:
import logging
# Suppress INFO-level logs for the entire logger
logging.getLogger().setLevel(logging.WARNING)

In [5]:
# define the figure path
figpath = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/zebrahub-multiome-analysis/figures/peak_umap_annotated/"
os.makedirs(figpath, exist_ok=True)
sc.settings.figdir = figpath

In [6]:
# import the peaks-by-celltype&timepoint pseudobulk object
# NOTE. the 2 MT peaks and 2 blacklisted peaks (since they go beyond the end of the chromosome) were filtered out.
adata_peaks = sc.read_h5ad("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/peaks_by_pb_annotated_master.h5ad")
adata_peaks

AnnData object with n_obs × n_vars = 640830 × 190
    obs: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'accessibility_neural_optic', 'accessibility_endoderm', 'accessibility_enteric_neurons', 'accessibility_notochord', 'accessibility_pharyngeal_arches', 'accessibility_floor_plate', 'accessibility_epidermis', 'accessibility_pronephros', 'accessibility_neural_floor_plate', 'accessibility_lateral_plate_mesoderm', 'accessibility_midbrain_hindbrain_boundary', 'accessibility_neural', 'accessibility_neurons', 'accessibility_neural_posterior', 'accessibility_neural_crest', 'accessibility_PSM', 'accessibility_fast_muscle', 'accessibility_endocrine_pancreas', 'accessibility_heart_myocardium', 'accessibility_neural_telencephalon', 'accessibility_optic_cup', 'accessibility_primordial_germ_cells', 'accessibility_differentiating_neurons', 'accessibility_muscle', 'accessibility_tail_bud', 'accessibility_hindbrain', 'accessibility_somites', 'accessibility_spinal_cord',

In [7]:
# read the annotation for the "peak_type_argelaguet"
filepath = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/all_peaks_annotated.csv"
annotated_peaks = pd.read_csv(filepath, index_col=0)
annotated_peaks.head()

,mean_counts,pct_dropout_by_counts,total_counts,accessibility_neural_optic,accessibility_endoderm,accessibility_enteric_neurons,accessibility_notochord,accessibility_pharyngeal_arches,accessibility_floor_plate,accessibility_epidermis,...,distance_to_tss,leiden_coarse,linked_gene,link_score,link_zscore,link_pvalue,associated_gene,association_type,log_total_counts,peak_type_argelaguet
n_cells_by_counts,,,,,,,,,,,,,,,,,,,,,
131,2.694737,31.052632,512.0,15.613128,10.325988,6.689899,2.991895,12.762588,12.170900,8.874177,...,11542.0,7,NaN,NaN,NaN,NaN,NaN,none,6.238325,intergenic
162,8.294737,14.736842,1576.0,57.789060,19.729371,42.719110,17.231317,22.624311,36.413397,30.458578,...,9107.0,12,NaN,NaN,NaN,NaN,NaN,none,7.362645,intergenic
170,18.636842,10.526316,3541.0,94.465606,30.845398,105.145162,41.163118,72.624480,72.500155,66.134746,...,8092.0,33,NaN,NaN,NaN,NaN,NaN,none,8.172164,intergenic
190,160.247368,0.000000,30447.0,568.299814,493.321985,480.077318,535.338964,641.692987,543.858841,572.097277,...,5953.0,26,NaN,NaN,NaN,NaN,rpl24,overlap,10.323743,exonic
172,33.905263,9.473684,6442.0,118.378152,102.449880,65.013071,141.397103,178.130543,108.393402,97.482937,...,2066.0,33,NaN,NaN,NaN,NaN,rpl24,overlap,8.770594,exonic


In [8]:
adata_peaks.obs["peak_type_argelaguet"] = annotated_peaks["peak_type_argelaguet"].values
adata_peaks.obs.head()

,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,accessibility_neural_optic,accessibility_endoderm,accessibility_enteric_neurons,accessibility_notochord,accessibility_pharyngeal_arches,accessibility_floor_plate,...,distance_to_tss,leiden_coarse,linked_gene,link_score,link_zscore,link_pvalue,associated_gene,association_type,leiden_unified,peak_type_argelaguet
1-32-526,131,2.694737,31.052632,512.0,15.613128,10.325988,6.689899,2.991895,12.762588,12.170900,...,11542.0,7,NaN,NaN,NaN,NaN,NaN,none,7_10,intergenic
1-2372-3057,162,8.294737,14.736842,1576.0,57.789060,19.729371,42.719110,17.231317,22.624311,36.413397,...,9107.0,12,NaN,NaN,NaN,NaN,NaN,none,12_10,intergenic
1-3427-4032,170,18.636842,10.526316,3541.0,94.465606,30.845398,105.145162,41.163118,72.624480,72.500155,...,8092.0,33,NaN,NaN,NaN,NaN,NaN,none,33_3,intergenic
1-4469-7268,190,160.247368,0.000000,30447.0,568.299814,493.321985,480.077318,535.338964,641.692987,543.858841,...,5953.0,26,NaN,NaN,NaN,NaN,rpl24,overlap,26_9,exonic
1-9541-9969,172,33.905263,9.473684,6442.0,118.378152,102.449880,65.013071,141.397103,178.130543,108.393402,...,2066.0,33,NaN,NaN,NaN,NaN,rpl24,overlap,33_6,exonic


In [26]:
# read the annotated peaks metadata, for celltype and timepoints (excluding the pseudobulk group with less than 20 cells)
annotated_peaks_ct_tp = pd.read_csv("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/all_peaks_annotated_ct_tp.csv",
                                    index_col=0)
annotated_peaks_ct_tp.head()

,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts,accessibility_neural_optic,accessibility_endoderm,accessibility_enteric_neurons,accessibility_notochord,accessibility_pharyngeal_arches,accessibility_floor_plate,...,log_accessibility_30somites,accessibility_lineage_CNS,accessibility_lineage_Neural Crest,accessibility_lineage_Paraxial Mesoderm,accessibility_lineage_Lateral Mesoderm,accessibility_lineage_Endoderm,accessibility_lineage_Epiderm,lineage,lineage_contrast,alpha_lineage
1-32-526,131,2.694737,31.052632,512.0,2.602188,1.720998,1.114983,0.598379,2.127098,2.028483,...,0.988233,1.968742,1.783009,1.422828,1.581245,1.385281,1.774835,CNS,2.255931,0.168038
1-2372-3057,162,8.294737,14.736842,1576.0,9.631510,3.288228,7.119852,3.446263,3.770718,6.068900,...,1.369809,6.618488,6.603891,3.357190,5.450942,3.149665,6.091716,CNS,1.189529,0.100000
1-3427-4032,170,18.636842,10.526316,3541.0,15.744268,5.140900,17.524194,8.232624,12.104080,12.083359,...,1.490621,14.508063,15.223006,10.480380,10.437248,6.229211,13.226949,Neural Crest,1.490091,0.109266
1-4469-7268,190,160.247368,0.000000,30447.0,94.716636,82.220331,80.012886,107.067793,106.948831,90.643140,...,4.281236,101.637955,111.745170,124.179247,118.166884,92.164870,98.399845,Paraxial Mesoderm,2.113717,0.157125
1-9541-9969,172,33.905263,9.473684,6442.0,19.729692,17.074980,10.835512,28.279421,29.688424,18.065567,...,2.884572,20.132878,23.678269,31.431252,24.344408,21.791484,19.496587,Paraxial Mesoderm,5.022713,0.380367


In [27]:
annotated_peaks_ct_tp.columns

Index(['n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts',
       'total_counts', 'accessibility_neural_optic', 'accessibility_endoderm',
       'accessibility_enteric_neurons', 'accessibility_notochord',
       'accessibility_pharyngeal_arches', 'accessibility_floor_plate',
       'accessibility_epidermis', 'accessibility_pronephros',
       'accessibility_neural_floor_plate',
       'accessibility_lateral_plate_mesoderm',
       'accessibility_midbrain_hindbrain_boundary', 'accessibility_neural',
       'accessibility_neurons', 'accessibility_neural_posterior',
       'accessibility_neural_crest', 'accessibility_PSM',
       'accessibility_fast_muscle', 'accessibility_endocrine_pancreas',
       'accessibility_heart_myocardium', 'accessibility_neural_telencephalon',
       'accessibility_optic_cup', 'accessibility_differentiating_neurons',
       'accessibility_muscle', 'accessibility_tail_bud',
       'accessibility_hindbrain', 'accessibility_somites',
       'accessibility

In [28]:
# copy over the metadata
adata_peaks.obs["celltype"] = adata_peaks.obs_names.map(annotated_peaks_ct_tp["celltype"])
adata_peaks.obs["celltype_contrast"] = adata_peaks.obs_names.map(annotated_peaks_ct_tp["celltype_contrast"])
adata_peaks.obs["timepoint"] = adata_peaks.obs_names.map(annotated_peaks_ct_tp["timepoint"])
adata_peaks.obs["timepoint_contrast"] = adata_peaks.obs_names.map(annotated_peaks_ct_tp["timepoint_contrast"])
adata_peaks.obs["lineage"] = adata_peaks.obs_names.map(annotated_peaks_ct_tp["lineage"])
adata_peaks.obs["lineage_contrast"] = adata_peaks.obs_names.map(annotated_peaks_ct_tp["lineage_contrast"])

In [30]:
# Create your DataFrame with metadata
df = pd.DataFrame(index=adata_peaks.obs_names)
# Add 3D UMAP coordinates
umap_3d = adata_peaks.obsm["X_umap_3D"]
df["UMAP_1"] = umap_3d[:, 0]
df["UMAP_2"] = umap_3d[:, 1]
df["UMAP_3"] = umap_3d[:, 2]

df["celltype"] = adata_peaks.obs["celltype"]
df["timepoint"] = adata_peaks.obs["timepoint"]
df["lineage"] = adata_peaks.obs["lineage"]
df["peak_type"] = adata_peaks.obs["peak_type_argelaguet"]
df["chromosome"] = adata_peaks.obs["chrom"]
df["leiden_coarse"] = adata_peaks.obs["leiden_coarse"]
df["leiden_fine"] = adata_peaks.obs["leiden_unified"]

# Export to CSV
df.to_csv("peak_umap_3d_annotated_v6.csv", index=True)

# Optional: Preview the DataFrame
print(f"DataFrame shape: {df.shape}")
print(df.head())

DataFrame shape: (640830, 10)
               UMAP_1    UMAP_2    UMAP_3        celltype  timepoint  \
1-32-526     2.265732  3.131183  5.916810  hatching_gland  10somites   
1-2372-3057 -1.087140 -0.566738  1.295660  hemangioblasts  10somites   
1-3427-4032 -1.458421 -1.449511 -4.037519       optic_cup  10somites   
1-4469-7268 -2.652668 -6.334654  0.916718  hemangioblasts  10somites   
1-9541-9969 -3.476965 -2.898085 -3.452023             PSM  10somites   

                       lineage   peak_type chromosome leiden_coarse  \
1-32-526                   CNS  intergenic          1             7   
1-2372-3057                CNS  intergenic          1            12   
1-3427-4032       Neural Crest  intergenic          1            33   
1-4469-7268  Paraxial Mesoderm      exonic          1            26   
1-9541-9969  Paraxial Mesoderm      exonic          1            33   

            leiden_fine  
1-32-526           7_10  
1-2372-3057       12_10  
1-3427-4032        33_3  
1-4469

In [31]:
# save the master adata object for the peaks-by-pseudobulk with annotation
adata_peaks.write_h5ad("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/peaks_by_pb_annotated_master.h5ad")
